# GenAI-Assisted Geographic Imputation

This notebook documents the AI application developed to recover missing
operational-province information in the Campania Financial & ESG Forecasting
project.

The original dataset contained missing values in the operational headquarters
province field. To reduce information loss before company filtering and
segmentation, the project combined:

- web search;
- an LLM (`llama-3.1-8b-instant` via Groq);
- deterministic output validation;
- repeated queries;
- majority-vote aggregation.

The original company records are proprietary and are not included in this
repository. The code below is therefore presented as a reusable public
template and uses placeholder company information.

## 1. Why this application was needed

The project observed missing operational-province information for **1,304
records**.

Because province was required for geographic segmentation, the workflow tried
to recover this field before applying the subsequent missingness filters.

The objective was intentionally narrow: return the operational province when
supported by retrieved evidence, otherwise return `unknown`.

In [ ]:
import os
import re
from collections import Counter
from typing import Optional, TypedDict, Annotated

import pandas as pd

from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages


## 2. API-key configuration

No API key is stored in this repository.

To run the application locally, define the environment variable
`GROQ_API_KEY` before creating the LLM client.

In [ ]:
# Example:
# export GROQ_API_KEY="your_key_here"

if not os.getenv("GROQ_API_KEY"):
    print(
        "GROQ_API_KEY is not configured. "
        "Set it before running live LLM calls."
    )

## 3. Search tool and LLM

The original implementation used:

- DuckDuckGo for web search;
- `llama-3.1-8b-instant` through Groq;
- temperature `0.3` to favor relatively deterministic outputs.

In [ ]:
search = DuckDuckGoSearchRun()

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.3,
)

CAMPANIA_PROVINCES = [
    "Avellino",
    "Benevento",
    "Caserta",
    "Napoli",
    "Salerno",
]

## 4. Typed workflow state

LangGraph was used to structure the application state and preserve the
information exchanged during execution.

In [ ]:
class AgentState(TypedDict):
    company_data: dict
    messages: Annotated[list[AnyMessage], add_messages]
    final_province: Optional[str]
    error_log: Optional[str]

## 5. Output normalization

The LLM is constrained to return a single province name or `unknown`.
The normalization function rejects multi-word explanations, punctuation-heavy
answers and unsupported output formats.

In [ ]:
def normalize_province(text: str) -> str:
    if not text:
        return "unknown"

    value = text.strip().strip('"“”\'').strip()

    if value.lower() in {"unknown", "sconosciuto"}:
        return "unknown"

    if any(separator in value for separator in [" ", "\n", "\t"]):
        return "unknown"

    if not re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", value):
        return "unknown"

    return value[0].upper() + value[1:].lower()

## 6. Repeated search and majority voting

For each incomplete record, the application generates several semantically
similar but syntactically different search queries.

The original workflow used:

- up to **5 search attempts**;
- an early-stop consensus threshold of **2 matching province outputs**;
- majority voting when early consensus was not reached.

This design reduces the risk that a single noisy search result or stochastic
generation determines the final imputation.

In [ ]:
def recover_operational_province(state: AgentState):
    company_name = state["company_data"]["company_name"]
    tax_id = state["company_data"]["tax_id"]

    max_attempts = 5
    consensus_threshold = 2

    answers = []
    logs = []

    queries = [
        f"operational headquarters {company_name} tax id {tax_id} province local unit",
        f"operational office {company_name} province tax id {tax_id}",
        f"{company_name} {tax_id} operational headquarters province",
        f"{company_name} {tax_id} secondary office province",
        f"operational headquarters {company_name} {tax_id} province",
    ]

    final_province = "unknown"

    for query in queries[:max_attempts]:
        raw_results = search.invoke(query)

        prompt = f"""
Retrieved evidence:
{raw_results}

Company: {company_name}
Tax ID: {tax_id}

Identify the PROVINCE of the OPERATIONAL HEADQUARTERS.

Rules:
- Prefer the operational headquarters over the legal headquarters.
- If multiple locations exist, prioritize one of:
  {CAMPANIA_PROVINCES}
- If the evidence is insufficient or uncertain, return exactly:
  unknown

Output requirements:
- Return ONLY the province name.
- No explanation, punctuation or additional text.
"""

        response = llm.invoke(prompt)
        raw_answer = (response.content or "").strip()
        answer = normalize_province(raw_answer)

        answers.append(answer)
        logs.append(
            {
                "query": query,
                "raw_answer": raw_answer,
                "normalized_answer": answer,
            }
        )

        counts = Counter(
            answer
            for answer in answers
            if answer != "unknown"
        )

        if counts:
            best_answer, frequency = counts.most_common(1)[0]

            if frequency >= consensus_threshold:
                final_province = best_answer
                break

    if final_province == "unknown":
        final_counts = Counter(
            answer
            for answer in answers
            if answer != "unknown"
        )

        if final_counts:
            final_province = final_counts.most_common(1)[0][0]

    return {
        "final_province": final_province,
        "messages": [HumanMessage(content=str(logs))],
    }

## 7. LangGraph workflow

The application uses a simple single-node graph:

**START → province-recovery node → END**

This is best described as a structured AI application rather than a full
ReAct agent, because the web-search step is explicitly required by the
workflow instead of being autonomously selected by the LLM.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

builder.add_node(
    "province_recovery",
    recover_operational_province,
)

builder.add_edge(START, "province_recovery")
builder.add_edge("province_recovery", END)

compiled_graph = builder.compile()

## 8. Example invocation

The live invocation is intentionally commented out so that opening the public
notebook does not trigger web searches or consume API credits.

In [ ]:
example_input = {
    "company_data": {
        "company_name": "EXAMPLE COMPANY SRL",
        "tax_id": "PLACEHOLDER",
    },
    "messages": [],
    "final_province": None,
    "error_log": None,
}

# result = compiled_graph.invoke(example_input)
# print(result["final_province"])

## 9. Validation and filtering

After the recovery stage, province values are normalized and checked against
the five Campania provinces.

Records outside the target geographic scope can then be excluded before the
downstream modelling pipeline.

In [ ]:
def is_campania_province(value):
    if pd.isna(value):
        return False

    normalized = str(value).strip().lower()

    accepted = {
        "napoli", "na",
        "avellino", "av",
        "benevento", "bn",
        "caserta", "ce",
        "salerno", "sa",
    }

    return normalized in accepted

## 10. Reliability considerations

The workflow includes several safeguards:

- web retrieval is performed before LLM synthesis;
- the prompt explicitly distinguishes operational from legal headquarters;
- outputs are constrained to a single validated province name;
- repeated searches use different query formulations;
- majority voting reduces dependence on a single generation;
- uncertain cases can remain `unknown` rather than being force-imputed.

These controls reduce, but do not eliminate, the risk of unsupported
imputations. Human verification remains appropriate for high-impact uses.

## Key Takeaways

This application demonstrates how Generative AI can support a broader
data-science pipeline without replacing deterministic data-quality controls.

Its role is deliberately narrow: recover a missing categorical attribute,
validate the result, and pass only structured output to the downstream
analysis.

The resulting province information supports:

- geographic segmentation;
- cluster-level model evaluation;
- regional economic analysis.

This notebook completes the public technical workflow of the
Campania Financial & ESG Forecasting project.